# Gold Layer — facts (Kimball, design 1.4)

Build the facts in `prod.gold` from Silver + Gold dims (run **after** `Analytic/gold/dims`).

| Fact | Type | Grain | Load (design 1.5) |
| --- | --- | --- | --- |
| `fact_video_daily_snapshot` | periodic snapshot | video × snapshot day | **partition overwrite** with `replaceWhere snapshot_date_key >= D` (atomic) |
| `fact_channel_daily_snapshot` | periodic snapshot | channel × snapshot day | same |
| `fact_comment` | transaction | comment | **MERGE** on `comment_id` |
| `fact_search_hit` | factless (event) | fetched_at × keyword × video | **insert-only MERGE** on the grain |

**Keys:** looked up in the Gold dims; no match → **−1** (Unknown), never NULL.

**Additivity:** `*_count` = semi-additive (take the last day for a period, never SUM across days); `*_gained` = additive (NULL on the first snapshot). Gains are vs the **previous snapshot row**, so they cover more than one day only where history has a gap (before the daily refresh started on 2026-10-26).


In [ ]:
%run ./_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"

print(f"Silver source: {SILVER}.stg_*")
print(f"Gold target: {GOLD}.fact_*")


def reload_from_key(table):
    """First snapshot_date_key to rebuild: the last loaded day (a rerun refreshes it) and every newer day."""
    if not spark.catalog.tableExists(table):
        return 0
    last = spark.sql(f"SELECT MAX(snapshot_date_key) AS k FROM {table}").collect()[0]["k"]
    return last or 0


## fact_video_daily_snapshot


In [ ]:
VIDEO_SNAP = f"{GOLD}.fact_video_daily_snapshot"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {VIDEO_SNAP} (
        snapshot_date_key INT,
        video_key         BIGINT,
        channel_key       BIGINT,
        view_count        BIGINT,
        like_count        BIGINT,
        comment_count     BIGINT,
        views_gained      BIGINT,
        likes_gained      BIGINT,
        comments_gained   BIGINT,
        video_age_days    INT,
        is_first_snapshot BOOLEAN,
        measured_at       TIMESTAMP,
        etl_date          TIMESTAMP
    )
    USING DELTA
    COMMENT 'Periodic snapshot: 1 video x 1 snapshot day (Vietnam time). Counts semi-additive, gains additive.'
""")

from_key = reload_from_key(VIDEO_SNAP)
video_snap_df = spark.sql(f"""
    WITH s AS (
        SELECT v.*,
               LAG(view_count)    OVER w AS prev_views,
               LAG(like_count)    OVER w AS prev_likes,
               LAG(comment_count) OVER w AS prev_comments,
               LAG(snapshot_date) OVER w AS prev_snapshot_date
        FROM {SILVER}.stg_videos v
        WINDOW w AS (PARTITION BY video_id ORDER BY snapshot_date)
    )
    SELECT
        CAST(date_format(s.snapshot_date, 'yyyyMMdd') AS INT)       AS snapshot_date_key,
        COALESCE(dv.video_key, -1)                                  AS video_key,
        COALESCE(dc.channel_key, -1)                                AS channel_key,
        s.view_count, s.like_count, s.comment_count,
        s.view_count - s.prev_views                                 AS views_gained,
        s.like_count - s.prev_likes                                 AS likes_gained,
        s.comment_count - s.prev_comments                           AS comments_gained,
        datediff(s.snapshot_date, to_date(from_utc_timestamp(s.published_at, '{SNAPSHOT_TZ}'))) AS video_age_days,
        s.prev_snapshot_date IS NULL                                AS is_first_snapshot,
        COALESCE(s.updated_at, s.created_at)                        AS measured_at,
        current_timestamp()                                         AS etl_date
    FROM s
    LEFT JOIN {GOLD}.dim_video dv   ON dv.video_id = s.video_id
    LEFT JOIN {GOLD}.dim_channel dc ON dc.channel_id = s.channel_id
    WHERE CAST(date_format(s.snapshot_date, 'yyyyMMdd') AS INT) >= {from_key}
""")
n = video_snap_df.count()
(video_snap_df.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"snapshot_date_key >= {from_key}")
    .saveAsTable(VIDEO_SNAP))
print(f"✅ {VIDEO_SNAP}: replaced snapshot_date_key >= {from_key} with {n} rows")


## fact_channel_daily_snapshot


In [ ]:
CHANNEL_SNAP = f"{GOLD}.fact_channel_daily_snapshot"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHANNEL_SNAP} (
        snapshot_date_key  INT,
        channel_key        BIGINT,
        subscriber_count   BIGINT,
        view_count         BIGINT,
        video_count        INT,
        subscribers_gained BIGINT,
        views_gained       BIGINT,
        videos_gained      INT,
        is_first_snapshot  BOOLEAN,
        measured_at        TIMESTAMP,
        etl_date           TIMESTAMP
    )
    USING DELTA
    COMMENT 'Periodic snapshot: 1 channel x 1 snapshot day (Vietnam time). Counts semi-additive, gains additive.'
""")

from_key = reload_from_key(CHANNEL_SNAP)
channel_snap_df = spark.sql(f"""
    WITH s AS (
        SELECT c.*,
               LAG(subscriber_count) OVER w AS prev_subscribers,
               LAG(view_count)       OVER w AS prev_views,
               LAG(video_count)      OVER w AS prev_videos,
               LAG(snapshot_date)    OVER w AS prev_snapshot_date
        FROM {SILVER}.stg_channels c
        WINDOW w AS (PARTITION BY channel_id ORDER BY snapshot_date)
    )
    SELECT
        CAST(date_format(s.snapshot_date, 'yyyyMMdd') AS INT)  AS snapshot_date_key,
        COALESCE(dc.channel_key, -1)                           AS channel_key,
        s.subscriber_count, s.view_count,
        CAST(s.video_count AS INT)                             AS video_count,
        s.subscriber_count - s.prev_subscribers                AS subscribers_gained,
        s.view_count - s.prev_views                            AS views_gained,
        CAST(s.video_count - s.prev_videos AS INT)             AS videos_gained,
        s.prev_snapshot_date IS NULL                           AS is_first_snapshot,
        COALESCE(s.updated_at, s.created_at)                   AS measured_at,
        current_timestamp()                                    AS etl_date
    FROM s
    LEFT JOIN {GOLD}.dim_channel dc ON dc.channel_id = s.channel_id
    WHERE CAST(date_format(s.snapshot_date, 'yyyyMMdd') AS INT) >= {from_key}
""")
n = channel_snap_df.count()
(channel_snap_df.write.format("delta").mode("overwrite")
    .option("replaceWhere", f"snapshot_date_key >= {from_key}")
    .saveAsTable(CHANNEL_SNAP))
print(f"✅ {CHANNEL_SNAP}: replaced snapshot_date_key >= {from_key} with {n} rows")


## fact_comment

Transaction fact, one row per comment. `comment_id` / `comment_text` are degenerate dimensions. Like / reply counts are overwritten with the latest value.


In [ ]:
COMMENT_FACT = f"{GOLD}.fact_comment"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {COMMENT_FACT} (
        comment_id      STRING,
        posted_date_key INT,
        video_key       BIGINT,
        channel_key     BIGINT,
        commenter_key   BIGINT,
        posted_at       TIMESTAMP,
        comment_text    STRING,
        comment_count   TINYINT,
        like_count      BIGINT,
        reply_count     INT,
        etl_date        TIMESTAMP
    )
    USING DELTA
    COMMENT 'Transaction fact: 1 comment. comment_count = 1, like_count / reply_count additive across comments.'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW fact_comment_src AS
    SELECT
        c.comment_id,
        CAST(date_format(from_utc_timestamp(c.published_at, '{SNAPSHOT_TZ}'), 'yyyyMMdd') AS INT) AS posted_date_key,
        COALESCE(dv.video_key, -1)      AS video_key,
        COALESCE(dc.channel_key, -1)    AS channel_key,
        COALESCE(dm.commenter_key, -1)  AS commenter_key,
        c.published_at                  AS posted_at,
        c.text                          AS comment_text,
        CAST(1 AS TINYINT)              AS comment_count,
        c.like_count,
        c.total_reply_count             AS reply_count,
        current_timestamp()             AS etl_date
    FROM {SILVER}.stg_comments c
    LEFT JOIN {GOLD}.dim_video dv      ON dv.video_id = c.video_id
    LEFT JOIN {GOLD}.dim_channel dc    ON dc.channel_id = dv.channel_id
    LEFT JOIN {GOLD}.dim_commenter dm  ON dm.commenter_key = xxhash64(COALESCE(c.author_channel_id, 'name:' || c.author_display_name))
""")
spark.sql(f"""
    MERGE INTO {COMMENT_FACT} AS t
    USING fact_comment_src AS s ON t.comment_id = s.comment_id
    WHEN MATCHED AND NOT (t.like_count <=> s.like_count AND t.reply_count <=> s.reply_count
                          AND t.video_key = s.video_key AND t.channel_key = s.channel_key
                          AND t.commenter_key = s.commenter_key AND t.comment_text <=> s.comment_text)
        THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"✅ {COMMENT_FACT}: {spark.table(COMMENT_FACT).count()} rows")


## fact_search_hit

Factless event: "this video came back for this keyword in this search run". `fetched_at` identifies the run. `search_order` / `result_rank` are saved by the app since 2026-10-26 (decision from 1.2); older runs have NULL.


In [ ]:
SEARCH_FACT = f"{GOLD}.fact_search_hit"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SEARCH_FACT} (
        fetched_at       TIMESTAMP,
        keyword_key      BIGINT,
        video_key        BIGINT,
        fetched_date_key INT,
        channel_key      BIGINT,
        search_order     STRING,
        result_rank      INT,
        hit_count        TINYINT,
        etl_date         TIMESTAMP
    )
    USING DELTA
    COMMENT 'Factless event: 1 video x 1 keyword x 1 search run. hit_count = 1 (additive), result_rank non-additive.'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW fact_search_hit_src AS
    WITH s AS (
        SELECT
            r.fetched_at,
            COALESCE(dk.keyword_key, -1)  AS keyword_key,
            COALESCE(dv.video_key, -1)    AS video_key,
            CAST(date_format(from_utc_timestamp(r.fetched_at, '{SNAPSHOT_TZ}'), 'yyyyMMdd') AS INT) AS fetched_date_key,
            COALESCE(dc.channel_key, -1)  AS channel_key,
            r.search_order,
            r.result_rank,
            CAST(1 AS TINYINT)            AS hit_count,
            current_timestamp()           AS etl_date,
            ROW_NUMBER() OVER (
                PARTITION BY r.fetched_at, COALESCE(dk.keyword_key, -1), COALESCE(dv.video_key, -1)
                ORDER BY r.id) AS rn
        FROM {SILVER}.stg_search_results r
        LEFT JOIN {GOLD}.dim_keyword dk ON dk.keyword_normalized = normalize_keyword(r.search_keyword)
        LEFT JOIN {GOLD}.dim_video dv   ON dv.video_id = r.video_id
        LEFT JOIN {GOLD}.dim_channel dc ON dc.channel_id = r.channel_id
    )
    SELECT * EXCEPT (rn) FROM s WHERE rn = 1
""")
spark.sql(f"""
    MERGE INTO {SEARCH_FACT} AS t
    USING fact_search_hit_src AS s
    ON t.fetched_at = s.fetched_at AND t.keyword_key = s.keyword_key AND t.video_key = s.video_key
    WHEN NOT MATCHED THEN INSERT *
""")
print(f"✅ {SEARCH_FACT}: {spark.table(SEARCH_FACT).count()} rows")


In [ ]:
# --- Verify ---
for t in ["fact_video_daily_snapshot", "fact_channel_daily_snapshot", "fact_comment", "fact_search_hit"]:
    print(f"📊 {GOLD}.{t}: {spark.table(f'{GOLD}.{t}').count()} rows")
print("\n--- Gold facts complete (run Analytic/gold/checks next) ---")
